# GEUVADIS transcriptional landscape via PCCA

Replication of Brown, Bray & Pachter, *Expression reflects population structure*
(PLOS Genetics 2018).

The claim: PCA of expression does **not** cluster individuals by population, while PCA of
genotype does. PCCA recovers the structure from expression by coupling the two views through
canonical correlation.

Reference values: first two canonical correlations **0.963** and **0.766**.

See `docs/4_pcca_methods.md` and `docs/superpowers/specs/2026-07-29-geuvadis-pcca-design.md`.

In [1]:
import sys
from pathlib import Path

# The notebook lives in playground/, so its own directory is the import root.
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd

from pcca import paths
from pcca.core import cca, loo_projection, orthonormalize, pca
from pcca.expression import build_expression_matrix, load_gd660
from pcca.figures import scatter_by_population
from pcca.genotype import load_eigenvec, restrict_to_panel
from pcca.metadata import load_sdrf, production_cohort, replication_cohort

## 1. Cohort

The paper's 343 individuals are recovered exactly from the SDRF: non-CEU, with Phase 1
genotypes. We then intersect with the Phase 3 panel, which carries only the 2504 unrelated
samples and omits 15 of them.

In [2]:
sdrf = load_sdrf(paths.SDRF)
paper_cohort = replication_cohort(sdrf)
print(f"paper's cohort:            {len(paper_cohort)}")

psam = paths.GENOTYPE / "work" / "phase3_orig.psam"
cohort = restrict_to_panel(paper_cohort, psam)
print(f"present in Phase 3 panel:  {len(cohort)}")
print(f"dropped (Phase 1 only):    {len(paper_cohort) - len(cohort)}")
print()
print(cohort["pop"].value_counts().to_string())

paper's cohort:            343
present in Phase 3 panel:  328
dropped (Phase 1 only):    15

TSI    90
FIN    86
GBR    77
YRI    75


In [3]:
ann, expr = load_gd660(paths.GD660_GENE_RPKM)
ann_f, X = build_expression_matrix(ann, expr, cohort)
print(f"expression matrix: {X.shape[0]} individuals x {X.shape[1]} genes")
print("paper reports 14,070 genes (GENCODE v27 protein-coding)")

geno_pcs = load_eigenvec(paths.GENOTYPE / "phase3" / "geuv_merged.eigenvec", cohort)
print(f"genotype PCs:      {geno_pcs.shape}")

expression matrix: 328 individuals x 12675 genes
paper reports 14,070 genes (GENCODE v27 protein-coding)
genotype PCs:      (328, 5)


## 2. Figure 1 — the negative result

Expression PCA should show populations intermixed. Genotype PCA should separate them.

In [4]:
u_x, _ = pca(X, dim=30)

scatter_by_population(
    u_x[:, :2], cohort,
    "Figure 1A - PCA of expression fails to cluster by population",
    "Expression PC1", "Expression PC2",
    paths.FIGURES / "pcca_fig1a_expression_pca.png",
)
scatter_by_population(
    geno_pcs[:, :2], cohort,
    "Figure 1B - PCA of genotype clusters by population",
    "Genotype PC1", "Genotype PC2",
    paths.FIGURES / "pcca_fig1b_genotype_pca.png",
)

# Quantify the contrast: how much variance in each leading component is
# explained by population label?
def variance_explained_by_population(coords, cohort):
    labels = cohort["pop"].to_numpy()
    total = coords.var(axis=0)
    within = np.zeros(coords.shape[1])
    for pop in np.unique(labels):
        m = labels == pop
        within += m.sum() / len(labels) * coords[m].var(axis=0)
    return 1 - within / total

print("variance explained by population label (PC1, PC2):")
print(f"  expression: {variance_explained_by_population(u_x[:, :2], cohort).round(3)}")
print(f"  genotype:   {variance_explained_by_population(geno_pcs[:, :2], cohort).round(3)}")

variance explained by population label (PC1, PC2):
  expression: [0.048 0.071]
  genotype:   [1.    0.975]


## 3. Figure 2A — PCCA, and the replication gate

**This is the gate.** The first two canonical correlations should approach 0.963 and 0.766.

In [5]:
u_y = orthonormalize(geno_pcs)
l_x, l_y, rho = cca(u_x, u_y, dim=2)

print(f"canonical correlations: {rho[0]:.3f}, {rho[1]:.3f}")
print("paper reports:          0.963, 0.766")

scatter_by_population(
    l_x, cohort,
    f"Figure 2A - PCCA projection of expression (rho = {rho[0]:.3f}, {rho[1]:.3f})",
    "Coordinate 1", "Coordinate 2",
    paths.FIGURES / "pcca_fig2a_projection.png",
)

print()
print("variance explained by population label (PCCA coords):")
print(f"  {variance_explained_by_population(l_x, cohort).round(3)}")

canonical correlations: 0.947, 0.782
paper reports:          0.963, 0.766

variance explained by population label (PCCA coords):
  [0.897 0.605]


## 4. Figure 2B — leave-one-out cross-validation

Each individual is dropped, both views' PCs and the canonical variables are refit on the
remainder, and the held-out individual is projected. Landing near their own population means
the structure is not an artefact of overfitting.

Uses the Gram-submatrix identity, so this is seconds rather than hours.

In [6]:
gram_y = geno_pcs @ geno_pcs.T
loo = loo_projection(X, gram_y, dim_x=30, dim_y=5, dim_z=2)

scatter_by_population(
    loo, cohort,
    "Figure 2B - leave-one-out projection",
    "Coordinate 1", "Coordinate 2",
    paths.FIGURES / "pcca_fig2b_loo.png",
)
print(f"leave-one-out projections: {loo.shape}")
print("variance explained by population label (held-out):")
print(f"  {variance_explained_by_population(loo, cohort).round(3)}")

leave-one-out projections: (328, 2)
variance explained by population label (held-out):
  [0.853 0.512]
